# Tutorial 05 - Exercise 01: Medical Dataset Preprocessing Pipeline

Demonstrates text standardization, missing value imputation, IQR outlier
capping, one-hot encoding, and feature scaling.

> Note: all imputations below use explicit column assignment
> (`df[col] = df[col].fillna(...)`) — the Pandas 3.x compatible form.
> Chained-assignment `fillna(..., inplace=True)` raises a `ChainedAssignmentError`
> under Pandas 2+ with Copy-on-Write, and is removed in Pandas 3.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/08_Data_Preprocessing_and_Modeling/` project folder (anchored on `data/messy_medical.csv`) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "08_Data_Preprocessing_and_Modeling"
ANCHOR_REL = "data/messy_medical.csv"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Load the Messy Medical Dataset


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(PROJ_DIR / "data" / "messy_medical.csv")
display(df)


### 2. Text Normalization & Imputation


In [ ]:
# Standardize text
for c in ["Gender", "Diagnosis"]:
    df[c] = df[c].astype(str).str.lower().str.strip().replace({"none": np.nan, "nan": np.nan})

# Mode imputation for categorical columns (Pandas 3.x safe: no chained inplace)
df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])
df["Diagnosis"] = df["Diagnosis"].fillna(df["Diagnosis"].mode()[0])

# Median imputation for numerical columns
num_cols = ["Age", "BloodPressure", "Cholesterol"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].median())

display(df)


### 3. Outlier Capping (IQR Method)


In [ ]:
for c in num_cols:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df[c] = df[c].clip(lower=lower, upper=upper)

display(df)


### 4. One-Hot Encoding & Standardization


In [ ]:
df_encoded = pd.get_dummies(df, columns=["Gender", "Diagnosis"], drop_first=True, dtype=int)

scaler = StandardScaler()
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])

print("--- Preprocessed Medical Dataset ---")
display(df_encoded)


### 5. Visual Exploration of the Cleaned Dataset (TP5 Exercise 01, Task 2)

The handout's Task 2 asks for four Seaborn charts on the cleaned medical
table. They are drawn on the frame *after* imputation but *before* capping
and scaling, because the charts are diagnostic: their whole point is to show
the outliers and the group structure that motivate the preprocessing steps
above. Each chart is followed by one sentence of reading, so the figure and
the written takeaway stay together.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
print("chart frame:", df.shape, "| missing:", int(df.isna().sum().sum()))

#### Chart 1 - Age vs BloodPressure, coloured by Gender

The scatter of `Age` against `BloodPressure` with `hue="Gender"` checks the
imputed gender labels: the two clouds overlap completely, so sex does not
separate blood pressure on this table, and the two points with blood pressure
above 190 are visibly outside the main band.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.scatterplot(data=df, x="Age", y="BloodPressure", hue="Gender", style="Gender",
                s=90, palette="Set2", ax=ax)
ax.set_title("Age vs Blood Pressure by Gender", fontweight="bold")
plt.tight_layout()
plt.show()

#### Chart 2 - Cholesterol distribution by Diagnosis

The distribution plot of `Cholesterol` split by `Diagnosis"` shows the three
groups side by side. The two hypertension patients sit at the top of the
range (300 and 350 before capping); healthy and diabetes overlap across the
rest - exactly what the capped upper bound of 317 will flatten.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.histplot(data=df, x="Cholesterol", hue="Diagnosis", multiple="stack",
             bins=8, palette="Set2", ax=ax)
ax.set_title("Cholesterol Distribution by Diagnosis", fontweight="bold")
plt.tight_layout()
plt.show()

#### Chart 3 - Combined numeric comparison by Diagnosis (`pd.melt`)

Comparing all three numeric columns at once needs the long form: `pd.melt`
stacks the three measurements into one `variable` column, and one box plot
then shows every group. The scales are wildly different (blood pressure around
120, cholesterol around 210, age around 52), so the box plot is read column by
column - the point of the chart is the *within-column* spread and the outliers,
not a comparison across columns.

In [ ]:
long = pd.melt(df, id_vars=["PatientID", "Diagnosis"], value_vars=num_cols,
               var_name="Measurement", value_name="Value")
display(long.head(6))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=long, x="Measurement", y="Value", hue="Diagnosis", palette="Set2", ax=ax)
ax.set_title("Numeric Measurements by Diagnosis (melted long form)", fontweight="bold")
plt.tight_layout()
plt.show()

#### Chart 4 - Mean BloodPressure by Diagnosis and Gender (`pivot_table` + heatmap)

The pivot table aggregates first (mean blood pressure per diagnosis and
gender) and the heatmap renders the small grid, with the missing combination
shown as a blank cell: no female hypertension patient exists in this
10-row table, and the heatmap makes that gap visible instead of hiding it.
The one hypertension group that exists (male) averages 205 - far above every
other cell of the grid.

In [ ]:
bp_pivot = df.pivot_table(index="Diagnosis", columns="Gender",
                         values="BloodPressure", aggfunc="mean").round(1)
display(bp_pivot)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(bp_pivot, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.5, ax=ax)
ax.set_title("Mean Blood Pressure by Diagnosis and Gender", fontweight="bold")
plt.tight_layout()
plt.show()

### 6. Outlier Treatment: Cap or Remove? (Discussion)

Both strategies are legitimate; they answer different questions and their
costs land in different places. On this table, applied to the same
imputed frame, the two versions diverge sharply:

| Criterion | Capping (`clip` to the IQR fence) | Removal (drop rows outside the fence) |
| :--- | :--- | :--- |
| Rows kept | 10 of 10 - every patient stays | 6 of 10 - P003, P005, P006 and P010 are dropped |
| Effect on the column | The 4 outlying values move to the fence (Age 120 -> 87, BP 210/200 -> 144/144, Chol 350 -> 318) | The quartiles themselves shift, because 40% of the rows left the table |
| Information | Preserved, distorted - a capped value says "at or beyond the fence" | Destroyed for the whole row, including its clean columns |
| When it fits | Skewed-but-real measurements (blood pressure spikes exist clinically), small samples | Confirmed entry errors, or when downstream models cannot tolerate any extreme value |

On a 10-row clinical table removal is the wrong default: dropping four
patients to protect three numbers changes the study population, not the
measurement error. The notebook keeps capping (Section 3) because the
outliers are plausible clinical extremes - and the price is that no
post-capping patient can exceed the fence, which the Cholesterol chart
above shows directly. One further caveat: with 10 observations the quartiles
themselves are fragile, so the fence positions move a lot if one row
changes; on a real cohort the same analysis would be run on hundreds of
rows before the choice means much.